文档分割：
- 基于字符分割
- 基于token分割

基于字符分割：字符分割 / 递归字符分割
- 短句分割
- 长文本分割

In [6]:
### 短句分割
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from rich.jupyter import display

chunk_size = 20 # 设置块大小
chunk_overlap = 10  # 设置块重叠大小

# 初始化递归字符文本分割器
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

# 初始化字符文本分割器
c_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separator='，',  # 默认以换行符作为分隔符
)

In [2]:
# 测试文本
text = "在AI的研究中，由于大模型规模非常大，模型参数很多，在大模型上跑完来验证参数好不好训练时间成本很高，所以一般会在小模型上做消融实验来验证哪些改进是有效的再去大模型上做实验。"

In [3]:
# 使用递归字符分割
r_splitter.split_text(text)

['在AI的研究中，由于大模型规模非常大，模',
 '大模型规模非常大，模型参数很多，在大模型',
 '型参数很多，在大模型上跑完来验证参数好不',
 '上跑完来验证参数好不好训练时间成本很高，',
 '好训练时间成本很高，所以一般会在小模型上',
 '所以一般会在小模型上做消融实验来验证哪些',
 '做消融实验来验证哪些改进是有效的再去大模',
 '改进是有效的再去大模型上做实验。']

In [7]:
# 使用字符分割
c_splitter.split_text(text)

Created a chunk of size 23, which is longer than the specified 20


['在AI的研究中，由于大模型规模非常大',
 '由于大模型规模非常大，模型参数很多',
 '在大模型上跑完来验证参数好不好训练时间成本很高',
 '所以一般会在小模型上做消融实验来验证哪些改进是有效的再去大模型上做实验。']

In [9]:
### 长文本分割
some_text = """When writing documents, writers will use document structure to group content. \
This can convey to the reader, which idea's are related. For example, closely related ideas \
are in sentances. Similar ideas are in paragraphs. Paragraphs form a document. \n\n  \
Paragraphs are often delimited with a carriage return or two carriage returns. \
Carriage returns are the "backslash n" you see embedded in this string. \
Sentences have a period at the end, but also, have a space.\
and words are separated by space."""

print(len(some_text))

496


In [10]:
c_splitter = CharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=0,
    separator=' '
)

# 递归字符分割器中，依次传入分隔符列表【双换行符、单换行符、空格、空字符】
# 分割时，先采用双换行符进行分割，如果此时切出的段落长度小于设置的chunk_size，则返回
# 如果某个段落依旧太长，继续用单换行符对其进行切分……依次使用分隔符列表切分直至长度满足要求
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=0,
    separators=["\n\n", "\n", " ", ""]
)

In [11]:
c_splitter.split_text(some_text)

['When writing documents, writers will use document structure to group content. This can convey to the reader, which idea\'s are related. For example, closely related ideas are in sentances. Similar ideas are in paragraphs. Paragraphs form a document. \n\n Paragraphs are often delimited with a carriage return or two carriage returns. Carriage returns are the "backslash n" you see embedded in this string. Sentences have a period at the end, but also,',
 'have a space.and words are separated by space.']

In [12]:
r_splitter.split_text(some_text)

["When writing documents, writers will use document structure to group content. This can convey to the reader, which idea's are related. For example, closely related ideas are in sentances. Similar ideas are in paragraphs. Paragraphs form a document.",
 'Paragraphs are often delimited with a carriage return or two carriage returns. Carriage returns are the "backslash n" you see embedded in this string. Sentences have a period at the end, but also, have a space.and words are separated by space.']

In [14]:
# 增加按句子分割
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=0,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)
r_splitter.split_text(some_text)

["When writing documents, writers will use document structure to group content. This can convey to the reader, which idea's are related. For example,",
 'closely related ideas are in sentances. Similar ideas are in paragraphs. Paragraphs form a document.',
 'Paragraphs are often delimited with a carriage return or two carriage returns. Carriage returns are the "backslash n" you see embedded in this',
 'string. Sentences have a period at the end, but also, have a space.and words are separated by space.']

基于token分割

In [5]:
# 由于llm的上下文窗口长度限制是按照token计数的，因此基于token分割可能更好

from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(
    chunk_size=1,
    chunk_overlap=0,
)

text = "foo bar bazzyfoo"
token_splitter.split_text(text)

['foo', ' bar', ' b', 'az', 'zy', 'foo']

分割Markdown文档

In [1]:
### Markdown文档本身就具有可分割的结构，可以根据标题或子标题分割，并将标题作为元数据添加到每个块中

from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_document = """
Title\n\n \
## 第一章\n\n \
李白乘舟将欲行\n\n 忽然岸上踏歌声\n\n \
### Section \n\n \
桃花潭水深千尺 \n\n
## 第二章\n\n \
不及汪伦送我情
"""

In [2]:
from IPython.display import Markdown

display(Markdown(markdown_document))


Title

 ## 第一章

 李白乘舟将欲行

 忽然岸上踏歌声

 ### Section 

 桃花潭水深千尺 

 
## 第二章

 不及汪伦送我情


In [4]:
# 定义想要分割的标题列表和名称
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
)

md_header_splits = markdown_splitter.split_text(markdown_document)
for s in range(len(md_header_splits)):
    print(f"第 {s} 个块：\n  {md_header_splits[s]}")

第 0 个块：
  page_content='Title'
第 1 个块：
  page_content='李白乘舟将欲行  
忽然岸上踏歌声' metadata={'Header 2': '第一章'}
第 2 个块：
  page_content='桃花潭水深千尺' metadata={'Header 2': '第一章', 'Header 3': 'Section'}
第 3 个块：
  page_content='不及汪伦送我情' metadata={'Header 2': '第二章'}
